# First-of-month: is it a VOL event (straddle) or a DRIFT (call)? — token-free (Colab)

Tests whether the day-1 effect is tradeable with options. Three questions, each answered with numbers:
1. **Is day-1 / the centered week actually more VOLATILE** (realized vol) than a normal day/week? (If not,
   a straddle is dead on arrival — vol is what a straddle needs.)
2. **Is the move symmetric (→ straddle) or one-sided up (→ call)?** win-rate, mean, median, skew, tails.
3. **Drift vs BREAKEVEN:** how does the expected move compare to a realistic ~1-week ATM straddle breakeven?
   (Long options must clear premium + theta + the vol-risk-premium.)

Run top-to-bottom; last cell exports one file. yfinance→Stooq fallback.


## capture setup (tees prints for the export cell)


In [ ]:
import builtins as _bi
REPORT_LINES = []; _realprint = _bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np
def load_spx():
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    try:
        df = yf.download('^GSPC', start='1990-01-01', progress=False, auto_adjust=True)
        if len(df):
            s = df['Close']; s = s.iloc[:,0] if hasattr(s,'columns') else s
            return pd.Series(np.asarray(s).ravel(), index=df.index, name='spx').dropna()
    except Exception as e: print('yf failed -> Stooq:', e)
    import urllib.request, io
    raw = urllib.request.urlopen('https://stooq.com/q/d/l/?s=^spx&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].rename('spx').dropna()
spx = load_spx()
d = pd.DataFrame({'px': spx}); d['ret'] = d['px'].pct_change(); d = d.dropna()
d['ym'] = d.index.to_period('M'); g = d.groupby('ym')
d['dom_start'] = g.cumcount() + 1
d['dom_end']   = g['px'].transform('size') - g.cumcount() - 1
print('SPX', len(d), 'days', d.index[0].date(), '->', d.index[-1].date())


## Q1 — is day-1 actually more VOLATILE? (realized vol / avg |move|)


In [ ]:
day1 = d[d['dom_start']==1]['ret']
d13  = d[d['dom_start'].isin([1,2,3])]['ret']
rest = d[d['dom_start']>3]['ret']
def volrow(r,label):
    return f'  {label:>16}: std {1e4*r.std():6.1f}bp | avg|move| {1e4*r.abs().mean():6.1f}bp | mean {1e4*r.mean():+6.1f}bp (n={len(r)})'
print('REALIZED VOL / move size:')
print(volrow(day1,'day 1')); print(volrow(d13,'days 1-3')); print(volrow(rest,'day 4+ (baseline)'))
print('\nIf day-1 std / avg|move| is NOT above baseline, it is a DRIFT day, not a VOL day -> straddle has no fuel.')


## Q2 — symmetric (straddle) or one-sided up (call)? distribution + tails


In [ ]:
try:
    from scipy import stats as _st
    def _skew(r): return _st.skew(r) if len(r)>3 else float('nan')
except Exception:
    def _skew(r):   # pandas skew fallback if scipy absent
        try: return float(pd.Series(r).skew())
        except Exception: return float('nan')
def dist(r,label):
    return (f'  {label:>10}: win {100*(r>0).mean():4.0f}% | mean {1e4*r.mean():+6.1f} | median {1e4*r.median():+6.1f} '
            f'| skew {_skew(r):+.2f} | best {1e2*r.max():+.1f}% worst {1e2*r.min():+.1f}%')
print(dist(day1,'day 1')); print(dist(rest,'baseline'))
print('\nHigh win% + positive mean/median + not-fat-left = DIRECTIONAL (call), not straddle.')


## Q3 — the centered week: drift vs a realistic straddle BREAKEVEN


In [ ]:
# 'week centered on day 1' = last 2 days of prior month + first 3 of this month (5 trading days)
d['nextym'] = d['ym']
moves=[]
idx = d.index
for i in range(len(d)):
    if d['dom_start'].iat[i]==1:
        lo=max(0,i-2); hi=min(len(d),i+3)   # 2 before day1 (prev month tail) .. day1,2,3
        w = d['ret'].iloc[lo:hi]
        moves.append((idx[i].date(), (1+w).prod()-1, len(w)))
mv = pd.DataFrame(moves, columns=['month','cum','n']); mv=mv[mv['n']==5]
ann_vol = d['ret'].std()*np.sqrt(252)
wk_sigma = d['ret'].std()*np.sqrt(5)
straddle_be = 0.8*wk_sigma      # rough ATM 1-wk straddle breakeven ~0.8*sigma_period (long premium)
print(f'SPX annualized vol ~ {100*ann_vol:.1f}% | 5-day sigma ~ {100*wk_sigma:.2f}%')
print(f'Rough 1-week ATM straddle BREAKEVEN move (need |move| > this to profit): ~{100*straddle_be:.2f}%')
print(f'  -- and IV usually prints ABOVE realized (vol-risk-premium), so real breakeven is WORSE.')
print(f'\nCentered-week (5d) stats over {len(mv)} months:')
print(f'  mean drift  : {100*mv["cum"].mean():+.2f}%   (the CALL bet: must beat ~1% weekly ATM call cost)')
print(f'  mean |move| : {100*mv["cum"].abs().mean():.2f}%   (the STRADDLE bet: must beat ~{100*straddle_be:.2f}%)')
hit = (mv['cum'].abs() > straddle_be).mean()
# compare vs all random 5-day windows
allw = (1+d['ret']).rolling(5).apply(np.prod, raw=True)-1; allw=allw.dropna()
base_hit = (allw.abs() > straddle_be).mean()
print(f'  % of centered-weeks that CLEAR straddle breakeven: {100*hit:.0f}%  vs random 5d weeks: {100*base_hit:.0f}%')
print('\nVERDICT LOGIC: if mean|move| < breakeven AND centered-week hit% is not ABOVE random, the straddle')
print('has no edge; if mean drift << ~1% call cost, the long call loses too. Small drift -> underlying, not options.')


## ⬇️ EXPORT — run LAST: save results to a file


In [ ]:
from datetime import datetime
fname='first_of_month_options_report.txt'
hdr=['='*66,'FIRST-OF-MONTH OPTIONS TEST — export',
     f'SPX {d.index[0].date()} -> {d.index[-1].date()} | {len(d)} days',
     f'generated {datetime.now().strftime("%Y-%m-%d %H:%M")}','='*66,'']
open(fname,'w').write('\n'.join(hdr)+'\n'+'\n'.join(REPORT_LINES)+'\n')
_realprint('wrote',fname,'—',len(REPORT_LINES),'lines')
if not REPORT_LINES: _realprint('!! empty — run cells above first')
try:
    from google.colab import files; files.download(fname); _realprint('download started')
except Exception as e:
    _realprint('not in Colab / blocked:',e,'- grab it from the folder icon (left)')
